# RAG -- Retrieval Augmented Generation
Ground LLM responses in your own documents. Eliminates hallucination for domain Q&A.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Why RAG?

| Problem | Without RAG | With RAG |
|---|---|---|
| Knowledge cutoff | Knows nothing after training | Retrieved docs are current |
| Hallucination | Invents facts | Grounded in text |
| Domain knowledge | Generic answers | Your documents |
| Cost | Expensive fine-tune | Cheap to update doc store |

```
Documents -> Chunk -> Embed -> Vector Store
                                   |
User Query -> Embed -> Search -> Top-K chunks -> LLM -> Answer
```

In [ ]:
# Step 1: Document chunking
documents = {
    'lr.txt': 'Linear regression fits y=wx+b by minimising MSE. Ridge adds L2 penalty. Lasso adds L1. R-squared measures variance explained.',
    'cls.txt': 'Classification predicts categories. Logistic regression uses sigmoid. Cross-entropy loss. F1 = harmonic mean of precision and recall.',
    'nn.txt': 'Neural networks stack layers. ReLU max(0,x). Backprop computes gradients. Dropout prevents overfitting. Adam optimiser.',
}
def chunk(text, size=25, overlap=5):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(' '.join(words[i:i+size]))
        i += size - overlap
    return [c for c in chunks if len(c)>20]
all_chunks = []
for src, txt in documents.items():
    for c in chunk(txt):
        all_chunks.append({'source':src,'text':c})
print(f'Total chunks: {len(all_chunks)}')
for c in all_chunks[:3]:
    print(f'  [{c["source"]}] {c["text"][:60]}...')

In [ ]:
# Step 2: TF-IDF vector store
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
texts = [c['text'] for c in all_chunks]
vec = TfidfVectorizer(ngram_range=(1,2), max_features=300)
embs = vec.fit_transform(texts)
print(f'Vector store: {embs.shape[0]} chunks x {embs.shape[1]} dims')

In [ ]:
# Step 3: Query -> Retrieve
def retrieve(query, k=3):
    qv = vec.transform([query])
    scores = cosine_similarity(qv, embs).flatten()
    top = np.argsort(scores)[-k:][::-1]
    return [(all_chunks[i], scores[i]) for i in top if scores[i]>0]
def rag_demo(query):
    results = retrieve(query, 3)
    print(f'Query: "{query}"')
    print('Retrieved context:')
    for chunk, score in results:
        print(f'  [{score:.3f}] {chunk["text"][:70]}...')
    return results
rag_demo('What is the difference between Ridge and Lasso?')

## Production: ChromaDB + sentence-transformers

```python
# pip install chromadb sentence-transformers openai
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI

embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()
coll = client.create_collection('docs')
for i, chunk in enumerate(all_chunks):
    coll.add(ids=[str(i)], embeddings=[embedder.encode(chunk['text']).tolist()],
             documents=[chunk['text']])

def rag_pipeline(question):
    results = coll.query(query_embeddings=[embedder.encode(question).tolist()], n_results=3)
    context = '\n'.join(results['documents'][0])
    oai = OpenAI()
    resp = oai.chat.completions.create(model='gpt-4o-mini', messages=[
        {'role':'system','content':f'Answer using only this context:\n{context}'},
        {'role':'user','content': question}
    ])
    return resp.choices[0].message.content
```

## RAG Evaluation (RAGAS)

| Metric | What it checks |
|---|---|
| Context Precision | Are retrieved chunks actually relevant? |
| Context Recall | Are all relevant chunks retrieved? |
| Answer Faithfulness | Is the answer grounded in context? |
| Answer Relevancy | Does it address the question? |

---
**Your turn:** Add a 4th document about 'decision trees'. Does the retrieval find it?